# Exercise 03 - The evidence in a database: parameterized SQL reconciliation

**Project Aegis - Phase 1 (Data & Storage) - learning loop step 2**

### Why this exercise
In Exercise 02 the ledger was a Python list and the reconciliation was `sum(...)`. In the real
system the evidence lives in a **database** and the agent reconciles by running **SQL** against it -
specifically the query the Postgres MCP tool exposes (spec FR-MCP-1, whose own example is
`SELECT SUM(amount) FROM operating_ledger WHERE account_type = 'lease_deficit'`). This exercise moves
the ledger into a real table and rebuilds the audit as a **parameterized SQL `SUM`**.

### What you'll learn
- **Schema as an evidence store** - designing the ledger table; money as **integer cents** (exact).
- **Parameterized queries** - binding values, never string-formatting them into SQL.
- **SQL injection** - why it's catastrophic *here specifically*: an **LLM** writes these queries.
- **Reconciliation in SQL** - `SELECT SUM(...) WHERE account = ?` as the audit operation.

### A note on SQLite vs PostgreSQL
We use Python's built-in **`sqlite3`** so this runs with zero install. The production store is
**PostgreSQL** (FR-MCP-1). Everything you learn here transfers directly; the only differences are
noted inline (mainly the parameter placeholder: SQLite uses `?`, psycopg/Postgres uses `%s`). We'll
stand up real Postgres when we build `src/`.

### Requirements
`pip install faker` (to regenerate the ledger). `sqlite3` is stdlib.

## Setup - regenerate a ledger to store

We reuse Exercise 02's generator (given) and build a ledger with **two** accounts: the
`operating_lease_liability` rows anchored to JPMorgan's real claim, plus some unrelated
`accounts_payable` rows. The second account matters for the injection demo later - it gives the
`WHERE` filter something real to filter *out*.

In [1]:
import sqlite3, random, urllib.request, json
from decimal import Decimal
from faker import Faker

SEC_USER_AGENT = {"User-Agent": "Aegis Learning Project sunitsingh.bitsg@gmail.com"}

def _pull_claim(ticker="JPM", concept="Liabilities"):
    try:
        def get(u):
            return json.load(urllib.request.urlopen(urllib.request.Request(u, headers=SEC_USER_AGENT), timeout=60))
        tbl = get("https://www.sec.gov/files/company_tickers.json")
        cik = next(str(r["cik_str"]).zfill(10) for r in tbl.values() if r["ticker"].upper() == ticker.upper())
        facts = get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json")
        rec = max((e for e in facts["facts"]["us-gaap"][concept]["units"]["USD"] if e.get("form") == "10-K"), key=lambda e: e["end"])
        return {"entity": facts["entityName"], "concept": concept, "claimed_value": rec["val"],
                "provenance": {"form": rec["form"], "fy": rec.get("fy"), "accession": rec["accn"]}}
    except Exception as e:
        print("offline, recorded claim:", e)
        return {"entity": "JPMORGAN CHASE & CO", "concept": "Liabilities", "claimed_value": 4062462000000,
                "provenance": {"form": "10-K", "fy": 2025, "accession": "0001628280-26-008131"}}

def make_ledger(target_total, n, seed, account):
    rng = random.Random(seed); fake = Faker(); fake.seed_instance(seed)
    tc = int((target_total * 100).to_integral_value())
    w = [rng.random() for _ in range(n)]; s = sum(w)
    cents = [int(tc * x / s) for x in w]; cents[-1] += tc - sum(cents)
    return [{"txn_id": f"{account[:3].upper()}-{seed}-{i:05d}", "date": fake.date_between(start_date='-1y').isoformat(),
             "counterparty": fake.company(), "account": account, "amount": Decimal(cents[i]) / 100} for i in range(n)]

claim = _pull_claim()
CLAIMED = Decimal(claim["claimed_value"])
lease_rows = make_ledger(CLAIMED, 500, seed=42, account="operating_lease_liability")
ap_rows    = make_ledger(Decimal("88000000.00"), 50, seed=99, account="accounts_payable")
all_rows = lease_rows + ap_rows
print(f"{len(all_rows)} rows: {len(lease_rows)} lease (= ${CLAIMED:,}) + {len(ap_rows)} AP")

550 rows: 500 lease (= $4,062,462,000,000) + 50 AP


## Step 1 - Design the evidence table (given)

The schema *is* a design decision. Notes on the columns:
- `amount_cents INTEGER` - money as **integer cents**, never a float column. Integers are exact;
  a `REAL`/float money column is the database version of the `0.1 + 0.2` bug. *(In Postgres you'd
  use `NUMERIC(20,2)`, a true fixed-point decimal; SQLite has no such type, so integer cents is the
  portable, exact choice - and it reinforces the same lesson.)*
- `txn_id` is the **primary key** - idempotent re-loads, and a stable handle for provenance.
- `account` is indexed-in-spirit - it's the column the reconciliation filters on.

In [2]:
conn = sqlite3.connect(":memory:")   # swap for a file path, or Postgres, with no other code change
conn.execute("""
    CREATE TABLE ledger (
        txn_id        TEXT PRIMARY KEY,
        txn_date      TEXT NOT NULL,
        counterparty  TEXT NOT NULL,
        account       TEXT NOT NULL,
        amount_cents  INTEGER NOT NULL
    )
""")
conn.execute("CREATE INDEX idx_ledger_account ON ledger(account)")
print("schema created")

schema created


## Step 2 - Load the rows with a parameterized INSERT (gap 1)

To insert, we convert each `Decimal` dollar amount to integer cents and pass the values as **bound
parameters** - the `?` placeholders - using `executemany`. We never build the SQL string from the
data. (This is the same safety principle as Step 4, applied to writes.)

**Your task (gap 1):** finish the `executemany` call - the SQL has five `?` placeholders; pass the
rows as tuples in the same column order.

In [3]:
def to_cents(amount: Decimal) -> int:
    return int((amount * 100).to_integral_value())

def load_ledger(conn, rows):
    params = [(r["txn_id"], r["date"], r["counterparty"], r["account"], to_cents(r["amount"])) for r in rows]
    # ==== YOUR CODE HERE (start) ====
    # Use conn.executemany with an INSERT INTO ledger (...) VALUES (?, ?, ?, ?, ?) statement,
    # passing `params`. Then conn.commit().
    conn.executemany(
        "INSERT INTO ledger (txn_id, txn_date, counterparty, account, amount_cents) VALUES (?, ?, ?, ?, ?)",
        params
    )
    conn.commit()
    # raise NotImplementedError
    # ==== YOUR CODE HERE (end) ====

load_ledger(conn, all_rows)
(count,) = conn.execute("SELECT COUNT(*) FROM ledger").fetchone()
assert count == len(all_rows), f"expected {len(all_rows)} rows, got {count}"
print("PASS - loaded", count, "rows")

PASS - loaded 550 rows


## Step 3 - Reconcile with a parameterized `SUM` (gap 2)

This is the audit, now in SQL: sum the evidence for one account and compare to the claim. The account
name is supplied as a **bound parameter**, not formatted into the string. We sum integer cents and
convert the result back to a `Decimal` at the boundary.

> **Postgres difference:** the placeholder becomes `%s` (e.g. `WHERE account = %s`) and you'd pass
> params to `cur.execute(sql, (account,))`. The *concept* - bind, don't interpolate - is identical.

**Your task (gap 2):** write the parameterized query `SELECT SUM(amount_cents) FROM ledger WHERE
account = ?` and bind `account`.

In [12]:
def ledger_sum(conn, account: str) -> Decimal:
    """Total (as Decimal dollars) for one account, via a parameterized query."""
    # ==== YOUR CODE HERE (start) ====
    # 1) execute 'SELECT SUM(amount_cents) FROM ledger WHERE account = ?' with (account,) bound
    # 2) fetch the single value (it may be None if no rows match -> treat as 0)
    # 3) return it as Decimal dollars (cents / 100)
    cur = conn.execute(
        "SELECT SUM(amount_cents) FROM ledger WHERE account = ?",
        (account,)
    )
    row = cur.fetchone()
    cents = row[0] or 0
    return Decimal(cents) / 100
    # raise NotImplementedError
    # ==== YOUR CODE HERE (end) ====

# --- self-check ---
got = ledger_sum(conn, "operating_lease_liability")
assert got == CLAIMED, f"expected {CLAIMED}, got {got}"
assert ledger_sum(conn, "no_such_account") == Decimal("0"), "missing account should sum to 0"
print("PASS - lease account sums to exactly", f"${got:,}")

PASS - lease account sums to exactly $4,062,462,000,000


## Step 4 - Why parameterization is non-negotiable here: SQL injection

The unsafe way is to format the account into the string. Watch what a crafted account value does.
The cell below runs both versions against the **same** request for the lease account.

Why this matters *in Aegis specifically*: the Data-Tooling agent generates SQL from an **LLM**, and
the Postgres MCP tool (FR-MCP-1) is specified as exposing *parameterized* SQL for exactly this reason.
An LLM (or a malicious document it ingested) can emit a value like `x' OR '1'='1`. With string
formatting that **bypasses the `WHERE` filter** and sums *every* account - a wrong, ungrounded number
silently entering an audit. With binding, the value is treated as data and matches nothing.

In [13]:
def unsafe_sum(conn, account):
    sql = f"SELECT SUM(amount_cents) FROM ledger WHERE account = '{account}'"   # NEVER do this
    cents = conn.execute(sql).fetchone()[0] or 0
    return Decimal(cents) / 100

malicious = "x' OR '1'='1"
print("total of ALL accounts        :", f"${ledger_sum(conn, 'operating_lease_liability') + ledger_sum(conn, 'accounts_payable'):,}")
print("unsafe_sum(malicious)         :", f"${unsafe_sum(conn, malicious):,}  <- filter bypassed, leaked the full total")
print("ledger_sum(malicious) [safe]  :", f"${ledger_sum(conn, malicious):,}  <- treated as data, matches nothing")

total of ALL accounts        : $4,062,550,000,000
unsafe_sum(malicious)         : $4,062,550,000,000  <- filter bypassed, leaked the full total
ledger_sum(malicious) [safe]  : $0  <- treated as data, matches nothing


## Step 5 - End-to-end: reconcile via the database

Same finding as Exercise 02, but the evidence total now comes from SQL. We also build a *discrepant*
database to confirm the audit still flags a planted delta when the numbers live in a table.

In [14]:
def reconcile_via_sql(conn, claim, account="operating_lease_liability") -> dict:
    claimed = Decimal(claim["claimed_value"])
    total = ledger_sum(conn, account)
    delta = claimed - total
    return {"entity": claim["entity"], "concept": claim["concept"], "claimed_value": claimed,
            "ledger_total": total, "delta": delta,
            "status": "MATCH" if delta == 0 else "DISCREPANCY",
            "provenance": claim["provenance"]}

print("clean DB :", reconcile_via_sql(conn, claim)["status"], "delta", f"${reconcile_via_sql(conn, claim)['delta']:,}")

# build a discrepant database
conn2 = sqlite3.connect(":memory:")
conn2.execute("CREATE TABLE ledger (txn_id TEXT PRIMARY KEY, txn_date TEXT, counterparty TEXT, account TEXT, amount_cents INTEGER)")
load_ledger(conn2, make_ledger(CLAIMED - Decimal("125000000.00"), 500, seed=7, account="operating_lease_liability"))
f = reconcile_via_sql(conn2, claim)
print("dirty DB :", f["status"], "delta", f"${f['delta']:,}  (accn {f['provenance']['accession']})")

clean DB : MATCH delta $0
dirty DB : DISCREPANCY delta $125,000,000  (accn 0001628280-26-008131)


## Recap & what's next

You moved the evidence into a database and rebuilt the audit as SQL:
- a deliberate **schema** with money as exact integer cents,
- **parameterized** writes and reads - bind, never interpolate,
- a live demo of **SQL injection** bypassing a `WHERE` filter, and why that's acute when an **LLM**
  generates the query (the reason FR-MCP-1 mandates parameterized MCP tools),
- reconciliation as `SELECT SUM(...) WHERE account = ?`, matching the spec's own example.

**Exercise 04** wraps Exercises 01-03 behind the **`SourceConnector` contract** (ADR-0001): a single
declarative *descriptor* (`{company, lookback, forms}`) resolved by `resolve -> fetch -> normalize`,
so adding a data source is a new adapter, not a new pipeline - the scaling answer you asked for.

**Stretch goals (optional):**
1. Add a `loaded_at` timestamp column and make `load_ledger` idempotent with `INSERT OR REPLACE`.
2. Write `account_breakdown(conn)` returning `SUM` grouped by account (`GROUP BY`).
3. Point `sqlite3.connect` at a file (`aegis_ledger.db`) and inspect it with any SQLite browser -
   proof the 'database swap' really is a one-line change.